# Лабораторная работа: классификация ЭКГ. Часть 1 — датасет

Лабораторная работа состоит из двух ноутбуков в двух проектах:

1. **этот ноутбук**, `ecg-warehouse/notebooks/lab_dataset.ipynb`: вы загрузите открытый датасет PTB-XL в хранилище ЭКГ `ecg-warehouse`, опишете выборку своего варианта спецификацией `DatasetSpec`, соберёте датасет и экспортируете его в проект `ecg-ml`;
2. `ecg-ml/notebooks/lab_training.ipynb`: вы обучите нейросетевой классификатор на PyTorch Lightning, оцените его на тестовой выборке и попробуете улучшить базовую модель.

[PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/) содержит 21 799 двенадцатиканальных ЭКГ длительностью 10 с от 18 869 пациентов. Записи размечены кардиологами кодами стандарта SCP-ECG: диагностическими заключениями (`NORM`, `IMI`, `LVH`, …), заключениями о форме ЭКГ и о ритме (`SR`, `AFIB`, …).

Ноутбук выполняется в окружении проекта `ecg-warehouse`: установите его командой `uv sync` в корне репозитория и выберите ядро из `.venv`.

## Варианты

Запишите номер своего варианта в `VARIANT`, а его коды — в `CLASSES` **в том порядке, в каком они указаны в таблице**: этот порядок задаёт столбцы матрицы меток `y`, а во второй части — выходы сети и порядок классов в метриках.

| Вариант | Коды SCP-ECG (порядок классов) | Состояния |
|:-:|---|---|
| 1 | `NORM`, `AFIB` | норма, фибрилляция предсердий |
| 2 | `NORM`, `CLBBB`, `CRBBB` | норма, полная блокада левой и правой ножки пучка Гиса |
| 3 | `NORM`, `IMI`, `ASMI` | норма, нижний и переднеперегородочный инфаркт миокарда |
| 4 | `NORM`, `LVH`, `LAFB` | норма, гипертрофия левого желудочка, блокада передней ветви левой ножки |
| 5 | `NORM`, `1AVB`, `IRBBB` | норма, АВ-блокада I степени, неполная блокада правой ножки |
| 6 | `SR`, `AFIB`, `STACH`, `SBRAD` | синусовый ритм, фибрилляция предсердий, синусовая тахикардия и брадикардия |
| 7 | `NORM`, `PVC`, `AFIB` | норма, желудочковая экстрасистолия, фибрилляция предсердий |

В варианте 6 все классы — заключения о ритме, поэтому норму в нём представляет код `SR` (синусовый ритм), а не диагностическое заключение `NORM`.

Задача **многометочная** (multi-label): классы не взаимоисключающие, у одной записи может быть несколько кодов варианта сразу (например, `IMI` и `ASMI`) или ни одного.

In [19]:
import os
from pathlib import Path

# Пути в ноутбуке отсчитываются от корня репозитория, а Jupyter может запустить ноутбук из notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

### Параметры

Пути указываются относительно корня репозитория `ecg-warehouse`.

- `VARIANT` — номер вашего варианта;
- `CLASSES` — **коды вашего варианта** в порядке из таблицы, например для варианта 2: `CLASSES = ["NORM", "CLBBB", "CRBBB"]`;
- `PTB_XL_ARCHIVE` — архив с PTB-XL, `PTB_XL_PATH` — каталог, в который он распаковывается;
- `DATASET_NAME` — имя датасета в реестре хранилища;
- `DATASET_EXPORT_PATH` — куда экспортировать датасет для обучения: в каталог `data/` проекта `ecg-ml`, склонированного рядом с `ecg-warehouse`. Если вы склонировали `ecg-ml` в другое место, исправьте путь.

In [20]:
VARIANT = 6
CLASSES = ["SR", "AFIB", "STACH", "SBRAD"]

PTB_XL_ARCHIVE = Path("data/ptb-xl.zip")
PTB_XL_PATH = Path("data/ptb-xl")

DATASET_NAME = f"ptb-xl-variant-{VARIANT}"
DATASET_EXPORT_PATH = Path("../ML_HomeWork4_2/data") / DATASET_NAME

## 1. Данные

Скачайте архив `ptb-xl.zip` (около 1,8 ГБ) по ссылке <https://disk.yandex.ru/d/fplepF1sNN24LQ> и положите его в каталог `data/` этого репозитория. Каталог `data/` исключён из git, поэтому ни архив, ни распакованные записи, ни само хранилище не попадут в коммит.

Ячейка ниже распаковывает архив в `PTB_XL_PATH` (около 3,2 ГБ), если он ещё не распакован. Если распаковка прервалась, удалите `data/ptb-xl` и запустите ячейку снова.

In [21]:
from pathlib import Path

PTB_XL_ARCHIVE = Path(
    "/home/ksuny/study/lab4_ML/ML_HomeWork4_1/data/ptb-xl.zip"
)

PTB_XL_PATH = Path("data/ptb-xl")

In [22]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

PTB_XL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not (PTB_XL_PATH / "ptbxl_database.csv").is_file():
    with ZipFile(PTB_XL_ARCHIVE) as archive:
        metadata_path, = [
            name for name in archive.namelist()
            if Path(name).name == "ptbxl_database.csv"
            and "__MACOSX" not in Path(name).parts
        ]

        prefix = metadata_path.removesuffix("ptbxl_database.csv")

        members = [
            name for name in archive.namelist()
            if name in {
                metadata_path,
                prefix + "scp_statements.csv"
            }
            or name.startswith(prefix + "records500/")
        ]

        with TemporaryDirectory(dir=PTB_XL_PATH.parent) as staging:
            archive.extractall(staging, members)
            extracted = Path(staging) / prefix
            extracted.rename(PTB_XL_PATH)

In [23]:
from ecg_warehouse.api import WarehouseAPI
from ecg_warehouse.dataset import DatasetSpec, SplitSpec

## 2. Хранилище ЭКГ

`ecg-warehouse` хранит метаданные записей (`recordings`) и диагнозы (`diagnoses`) в таблицах Apache Iceberg, а сами сигналы — отдельно, в репозитории Icechunk. `WarehouseAPI` — точка входа для загрузки данных, SQL-запросов и сборки датасетов.

Конфигурация `cfg/local.json` описывает локальное хранилище в каталоге `data/`: каталог таблиц Iceberg — `data/catalog.db`, сигналы — `data/ECG/signals.icechunk`.

In [24]:
wh = WarehouseAPI("cfg/local.json")

### Загрузка PTB-XL

`ingest_ptb_xl` читает записи с частотой дискретизации 500 Гц (12 отведений × 5000 отсчётов) и их коды SCP-ECG. Загрузка всего датасета занимает заметное время. Если её прервать, повторный запуск пропустит уже загруженные записи.

In [25]:
wh.ingest_ptb_xl(data_dir=PTB_XL_PATH, rate=500)

100%|██████████████████████████████████████████████████████| 21799/21799 [00:00<00:00, 61308.87it/s]
  2026-11-13T20:22:14.366210Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

Signals store cleanup: freed 0.0 MiB


`wh.sql` выполняет SQL-запрос (DuckDB) к таблицам `recordings` и `diagnoses` и возвращает `pandas.DataFrame`. Проверим, сколько записей и пациентов загружено.

In [26]:
wh.sql(
    "SELECT source, sampling_rate_hz, num_leads, count(*) AS recordings, count(DISTINCT patient_id) AS patients FROM recordings GROUP BY ALL"
)

,source,sampling_rate_hz,num_leads,recordings,patients
0,ptb-xl-1.0.3,500,12,21799,18869


**Задание.** Посчитайте в ячейке ниже, сколько записей приходится на каждый класс вашего варианта, и оцените баланс классов. Например, для варианта 1:

```python
wh.sql("""
    SELECT code, label_text, count(DISTINCT recording_id) AS recordings
    FROM diagnoses
    WHERE code IN ('NORM', 'AFIB')
    GROUP BY ALL
    ORDER BY recordings DESC
""")
```

In [27]:
class_statistics = wh.sql("""
    SELECT
        code,
        label_text,
        count(DISTINCT recording_id) AS recordings
    FROM diagnoses
    WHERE code IN ('SR', 'AFIB', 'STACH', 'SBRAD')
      AND code_system = 'SCP-ECG'
    GROUP BY code, label_text
    ORDER BY CASE code
        WHEN 'SR' THEN 1
        WHEN 'AFIB' THEN 2
        WHEN 'STACH' THEN 3
        WHEN 'SBRAD' THEN 4
    END
""")

display(class_statistics)

,code,label_text,recordings
0,SR,sinus rhythm,16748
1,AFIB,atrial fibrillation,1514
2,STACH,sinus tachycardia,826
3,SBRAD,sinus bradycardia,637


### Баланс класса

In [28]:
class_statistics["percentage"] = (
    class_statistics["recordings"]
    / class_statistics["recordings"].sum() * 100
).round(2)

class_statistics

,code,label_text,recordings,percentage
0,SR,sinus rhythm,16748,84.91
1,AFIB,atrial fibrillation,1514,7.68
2,STACH,sinus tachycardia,826,4.19
3,SBRAD,sinus bradycardia,637,3.23


## 3. Сборка датасета

`DatasetSpec` декларативно описывает выборку: по спецификации и текущему состоянию таблиц датасет собирается воспроизводимо.

- `name` — имя датасета в реестре хранилища;
- `sources`, `num_leads`, `sampling_rate_hz`, `min_duration_sec`, `max_duration_sec` — фильтры записей: только PTB-XL, 12 отведений, 500 Гц, ровно 10 с;
- `code_systems`, `codes` — оставить только диагнозы SCP-ECG с кодами из `CLASSES`;
- `recording_columns`, `diagnosis_columns` — столбцы метаданных, которые попадут в датасет;
- `split` — разбиение на train и test **по пациентам**: все записи пациента попадают в одну часть, в test уходит около 20 % пациентов, `seed` фиксирует разбиение. Так записи одного человека не окажутся одновременно в обучении и в тесте.

In [29]:
dataset_spec = DatasetSpec(
    name=DATASET_NAME,
    sources=["ptb-xl-1.0.3"],
    num_leads=12,
    sampling_rate_hz=500,
    code_systems=["SCP-ECG"],
    codes=CLASSES,
    min_duration_sec=10,
    max_duration_sec=10,
    recording_columns=[
        "recording_id",
        "source",
        "patient_id",
        "sampling_rate_hz",
        "duration_sec",
        "age_at_recording",
        "sex",
    ],
    diagnosis_columns=[
        "diagnosis_id",
        "recording_id",
        "code",
        "code_system",
        "label_text",
    ],
    split=SplitSpec(test_fraction=0.2, seed=0),
)

`build_dataset` собирает датасет по спецификации и регистрирует его в реестре хранилища под именем и версией. Ячейка выводит манифест сборки: имя, версию, спецификацию и описание файлов. Сигналы не копируются — датасет ссылается на них в хранилище.

In [30]:
dataset = wh.build_dataset(spec=dataset_spec)
dataset

Built dataset ptb-xl-variant-6@0.0.0-dev.hd0aabe8f5ed69018 (dev): 21799 recordings, 19725 diagnoses


{'name': 'ptb-xl-variant-6',
 'version': '0.0.0-dev.hd0aabe8f5ed69018',
 'mode': 'dev',
 'release_type': None,
 'content_hash': 'd0aabe8f5ed69018',
 'spec': {'name': 'ptb-xl-variant-6',
  'sources': ['ptb-xl-1.0.3'],
  'num_leads': 12,
  'sampling_rate_hz': 500,
  'min_duration_sec': 10.0,
  'max_duration_sec': 10.0,
  'code_systems': ['SCP-ECG'],
  'codes': ['SR', 'AFIB', 'STACH', 'SBRAD'],
  'drop_recordings_with_other_codes': False,
  'require_diagnosis': False,
  'recording_columns': ['recording_id',
   'source',
   'patient_id',
   'sampling_rate_hz',
   'duration_sec',
   'age_at_recording',
   'sex'],
  'diagnosis_columns': ['diagnosis_id',
   'recording_id',
   'code',
   'code_system',
   'label_text'],
  'signals_format': 'icechunk',
  'split': {'test_fraction': 0.2, 'seed': 0}},
 'snapshots': {'recordings': 1319311970530113514,
  'diagnoses': 8129487224278041640},
 'artifacts': {'recordings': {'uri': 'file:///home/ksuny/study/lab4_ML/ML_HomeWork4_1/data/datasets/ptb-xl-varia

Экспорт в формат `npz` записывает в `DATASET_EXPORT_PATH`:

- `signals.npz` с массивами `X` — сигналы `float32` формы `(записи, 12, 5000)`; `y` — метки формы `(записи, len(CLASSES))`, где `y[i, j] = 1`, если у записи `i` есть код `CLASSES[j]`; а также `classes`, `recording_id` и `split` (`train` / `test`);
- `meta.csv` — метаданные записей, `diagnoses.csv` — их диагнозы.

In [31]:
wh.export_dataset(
    dataset_ref=f"{dataset['name']}@{dataset['version']}",
    format="npz",
    output=DATASET_EXPORT_PATH,
)

  0%|                                                                     | 0/21799 [00:00<?, ?it/s]  2026-11-13T20:22:15.151357Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-13T20:22:15.153449Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

100%|████████████████████████████████████████████████████████| 21799/21799 [00:23<00:00, 918.89it/s]


{'name': 'ptb-xl-variant-6',
 'version': '0.0.0-dev.hd0aabe8f5ed69018',
 'format': 'npz',
 'output': '/home/ksuny/study/lab4_ML/ML_HomeWork4_2/data/ptb-xl-variant-6',
 'subset': {'split': None, 'limit': None},
 'recordings': {'dataset': 21799, 'exported': 21799},
 'diagnoses': {'dataset': 19725, 'exported': 19725},
 'files': {'signals.npz': {'X': [21799, 12, 5000],
   'y': [21799, 4],
   'classes': ['SR', 'AFIB', 'STACH', 'SBRAD'],
   'recording_id': [21799],
   'split': [21799]},
  'meta.csv': {'rows': 21799},
  'diagnoses.csv': {'rows': 19725}}}

## Что дальше

1. Сохраните ноутбук вместе с результатами ячеек и закоммитьте его в свою ветку, как описано в инструкции к лабораторной работе.
2. Переходите ко второй части: `ecg-ml/notebooks/lab_training.ipynb`.